## Существующие SAE для open-source моделей:

| LLM | Готовые веса / источник | Где обучен SAE и размер словаря | Значение для нашего проекта |
| --- | --- | --- | --- |
| Pythia-70M-deduped, base | [ctigges/pythia-70m-deduped__res-sm_processed](https://huggingface.co/ctigges/pythia-70m-deduped__res-sm_processed); [реестр](https://decoderesearch.github.io/SAELens/latest/pretrained_saes/pythia-70m-deduped/) | Residual stream слоёв 0–5 и входные embeddings; 32 768 признаков | Кандидат для дешёвой проверки извлечения активаций; пригодность для диалогового refusal надо оценивать отдельно |
| GPT-2 small, base | [jbloom/GPT2-Small-SAEs-Reformatted](https://huggingface.co/jbloom/GPT2-Small-SAEs-Reformatted); [реестр](https://decoderesearch.github.io/SAELens/latest/pretrained_saes/gpt2-small/) | Residual stream: `resid_pre` слоёв 0–11 и `resid_post` последнего; 24 576 признаков | Небольшая модель для семантических экспериментов и отладки |
| Gemma 2 2B / 9B / 27B, base | [Gemma Scope](https://huggingface.co/google/gemma-scope); [статья](https://arxiv.org/abs/2408.05147); пример весов: [2B residual SAE](https://huggingface.co/google/gemma-scope-2b-pt-res) | JumpReLU; residual, attention, MLP; все слои 2B/9B, выбранные слои 27B; несколько ширин и L0 | 2B — кандидат для EDA концептов; не подменять base на IT без проверки переноса |
| Gemma 2 9B **IT** | [google/gemma-scope-9b-it-res](https://huggingface.co/google/gemma-scope-9b-it-res); [реестр](https://decoderesearch.github.io/SAELens/latest/pretrained_saes/gemma-2-9b-it/) | `resid_post`, слои 9, 20, 31; 16 384 / 131 072 признака | SAE именно для instruction-tuned модели, но LLM заметно крупнее 1–2B |
| Gemma 3 270M / 1B / 4B / 12B / 27B, **PT и IT** | [Gemma Scope 2](https://huggingface.co/google/gemma-scope-2); пример: [1B IT](https://huggingface.co/google/gemma-scope-2-1b-it) | SAE на residual/attention/MLP всех слоёв; расширенные наборы ширин/L0 на подмножестве слоёв; transcoders и crosscoders лежат отдельно | 1B IT — кандидат для поведенческих экспериментов с SAE, обученным на том же варианте LLM |
| Qwen3-1.7B-Base | [Qwen/SAE-Res-Qwen3-1.7B-Base-W32K-L0_50](https://huggingface.co/Qwen/SAE-Res-Qwen3-1.7B-Base-W32K-L0_50); [реестр](https://decoderesearch.github.io/SAELens/latest/pretrained_saes/qwen_qwen3-1.7b-base/) | `resid_post`, слои 0–27; 32 768 признаков; релизы L0=50 и 100 | Небольшая альтернатива для изучения концептов |
| Qwen3.5-2B-Base | [Qwen/SAE-Res-Qwen3.5-2B-Base-W32K-L0_50](https://huggingface.co/Qwen/SAE-Res-Qwen3.5-2B-Base-W32K-L0_50); [реестр](https://decoderesearch.github.io/SAELens/latest/pretrained_saes/qwen_qwen3.5-2b-base/) | TopK; `resid_post`, слои 0–23; `d_in=2048`, `d_sae=32768`; L0=50 или 100 | Близко к модели в `sae_from_scratch.ipynb`, но словарь обучен на **Base** |
| Qwen2.5-7B-Instruct | [andyrdt/saes-qwen2.5-7b-instruct](https://huggingface.co/andyrdt/saes-qwen2.5-7b-instruct); [реестр](https://decoderesearch.github.io/SAELens/latest/pretrained_saes/qwen_qwen2.5-7b-instruct/) | `resid_post`, слои 3, 7, 11, 15, 19, 23, 27; 131 072 признака | Вариант для поведенческого steering при большем бюджете |

## Эксперимент с Gemma 3 1B IT

Следуем разделу «Хуки» в [checkpoint-notes](01_sae_and_steering.md): загружаем [Gemma 3 1B IT](https://huggingface.co/google/gemma-3-1b-it), собираем три состояния residual stream одного слоя и применяем предобученный SAE к `resid_post`.

### 1. Загрузка модели

Ревизия весов закреплена. `snapshot_download` скачает модель в кеш Hugging Face, а при повторном запуске использует уже скачанные файлы. Для первого скачивания нужен доступ к Gemma на Hugging Face и вход через `hf auth login`.

In [22]:
from huggingface_hub import snapshot_download

MODEL_ID = "google/gemma-3-1b-it"
MODEL_REVISION = "dcc83ea841ab6100d6b47a070329e1ba4cf78752"

model_path = snapshot_download(
    MODEL_ID,
    revision=MODEL_REVISION,
    allow_patterns=["*.json", "*.model", "*.safetensors"],
)
print("Model:", model_path)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Model: /Users/akubaxx/.cache/huggingface/hub/models--google--gemma-3-1b-it/snapshots/dcc83ea841ab6100d6b47a070329e1ba4cf78752


In [23]:
from contextlib import contextmanager
from importlib.metadata import version

import pandas as pd
import torch
from IPython.display import display
from transformers import AutoModelForCausalLM, AutoTokenizer

LAYER = 13  # индексация с нуля
DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
MODEL_DTYPE = torch.bfloat16 if DEVICE == "cuda" else torch.float32
torch.manual_seed(42)

print(f"device={DEVICE}, model_dtype={MODEL_DTYPE}")
print({name: version(name) for name in ["torch", "transformers", "huggingface-hub"]})


device=mps, model_dtype=torch.float32
{'torch': '2.14.1', 'transformers': '5.18.0', 'huggingface-hub': '1.33.0'}


In [24]:
tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    local_files_only=True,
    dtype=MODEL_DTYPE,
    attn_implementation="eager",
).to(DEVICE).eval()
model.requires_grad_(False)

assert 0 <= LAYER < model.config.num_hidden_layers
block = model.model.layers[LAYER]
print(type(model).__name__, "ready")
print(f"layer={LAYER}, layers={model.config.num_hidden_layers}, hidden_size={model.config.hidden_size}")


Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

Gemma3ForCausalLM ready
layer=13, layers=26, hidden_size=1152


### 2. Снимаем активации руками

Выберем слой **13**, индексация с нуля. Названия `hook_resid_*` ниже обозначают точки из заметок; в Hugging Face мы сопоставляем им обычные PyTorch-хуки.

| Имя | Состояние | Точка в Hugging Face Gemma 3 |
| --- | --- | --- |
| `hook_resid_pre` | До attention и MLP слоя | Вход `model.layers[13]` |
| `hook_resid_mid` | После прибавления attention, до MLP | Вход `pre_feedforward_layernorm`, **до нормализации** |
| `hook_resid_post` | После прибавления MLP | Выход `model.layers[13]` |

Для начала делаем один forward по токенам пользовательского запроса с chat template. Получим три тензора формы `[batch, tokens, hidden_size]` с одинаковыми позициями токенов.


In [25]:
@contextmanager
def capture_residuals(block):
    """Собрать pre/mid/post за один forward, затем снять все хуки."""
    cache = {}

    def capture_pre(module, args, kwargs):
        # Transformers может передавать hidden_states именованным аргументом.
        hidden = args[0] if args else kwargs["hidden_states"]
        cache["hook_resid_pre"] = hidden.detach().clone()

    def capture_mid(module, args):
        cache["hook_resid_mid"] = args[0].detach().clone()

    def capture_post(module, args, output):
        hidden = output[0] if isinstance(output, tuple) else output
        cache["hook_resid_post"] = hidden.detach().clone()

    handles = []
    try:
        handles.append(block.register_forward_pre_hook(capture_pre, with_kwargs=True))
        handles.append(block.pre_feedforward_layernorm.register_forward_pre_hook(capture_mid))
        handles.append(block.register_forward_hook(capture_post))
        yield cache
    finally:
        for handle in handles:
            handle.remove()


In [26]:
messages = [{"role": "user", "content": "Explain in two sentences why the sky is blue."}]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt",
).to(DEVICE)

# Один проход по запросу, включая служебные токены chat template.
# model.model обходит LM head: для сбора активаций logits не нужны.
with torch.inference_mode(), capture_residuals(block) as activations:
    model.model(**inputs, use_cache=False)

token_ids = inputs["input_ids"][0]
print(tokenizer.decode(token_ids))


<bos><start_of_turn>user
Explain in two sentences why the sky is blue.<end_of_turn>
<start_of_turn>model



In [27]:
expected_shape = (*inputs["input_ids"].shape, model.config.hidden_size)
assert set(activations) == {"hook_resid_pre", "hook_resid_mid", "hook_resid_post"}

for sae_input in activations.values():
    assert tuple(sae_input.shape) == expected_shape
    assert torch.isfinite(sae_input).all()

activation_summary = pd.DataFrame([
    {
        "hook": name,
        "shape": tuple(hidden.shape),
        "dtype": str(hidden.dtype),
        "mean_token_l2": hidden.float().norm(dim=-1).mean().item(),
    }
    for name, hidden in activations.items()
])
display(activation_summary)


,hook,shape,dtype,mean_token_l2
0,hook_resid_pre,"(1, 19, 1152)",torch.float32,9177.464844
1,hook_resid_mid,"(1, 19, 1152)",torch.float32,9077.071289
2,hook_resid_post,"(1, 19, 1152)",torch.float32,9678.933594


### 3. Применяем предобученный SAE

Берём [Gemma Scope 2 для Gemma 3 1B IT](https://huggingface.co/google/gemma-scope-2-1b-it): `resid_post`, слой 13, 16 384 признака, L0 `medium`. Функция `get_sae_features(messages, top_k=5)` использует загруженные модель и SAE: применяет chat template, заново собирает активации и передаёт их в `sae.encode()` через [SAELens](https://decoderesearch.github.io/SAELens/latest/usage/).

В возвращённом `df` одна строка на входной токен, включая chat template, в исходном порядке. Колонки `top_1_feature` … `top_5_feature` содержат пары `(feature_id, activation)` пяти самых сильных положительных активаций по убыванию силы. Если активных признаков меньше пяти, оставшиеся ячейки пустые. `n_active` — общее число активных признаков до отбора top-5. ID обозначают координаты SAE; смысл признаков здесь ещё не интерпретируем.

In [39]:
from sae_lens import SAE

SAE_RELEASE = "gemma-scope-2-1b-it-res"
SAE_ID = f"layer_{LAYER}_width_16k_l0_medium"
SAE_HOOK = "hook_resid_post"

sae = SAE.from_pretrained(
    release=SAE_RELEASE,
    sae_id=SAE_ID,
    device=DEVICE,
    dtype="float32",
).eval()
sae.requires_grad_(False)

assert sae.cfg.metadata.model_name == MODEL_ID
assert sae.cfg.metadata.hook_name == f"blocks.{LAYER}.{SAE_HOOK}"
assert sae.cfg.d_in == model.config.hidden_size
print(f"sae-lens={version('sae-lens')}, SAE={SAE_RELEASE}/{SAE_ID}")
print(f"d_in={sae.cfg.d_in}, d_sae={sae.cfg.d_sae}")


sae-lens=6.55.0, SAE=gemma-scope-2-1b-it-res/layer_13_width_16k_l0_medium
d_in=1152, d_sae=16384


In [43]:
@torch.inference_mode()
def get_sae_features(messages, top_k=5):
    """Вернуть токены и top-k SAE-фичей, используя загруженные model, tokenizer и sae."""
    if not isinstance(top_k, int) or not 1 <= top_k <= sae.cfg.d_sae:
        raise ValueError(f"top_k must be an integer between 1 and {sae.cfg.d_sae}")

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)
    token_ids = inputs["input_ids"][0].tolist()

    with capture_residuals(block) as activations:
        model.model(**inputs, use_cache=False)

    sae_input = activations[SAE_HOOK]
    assert tuple(sae_input.shape) == (1, len(token_ids), sae.cfg.d_in)
    assert torch.isfinite(sae_input).all()
    sae_output = sae.encode(sae_input.to(device=sae.device, dtype=sae.dtype))
    assert tuple(sae_output.shape) == (1, len(token_ids), sae.cfg.d_sae)
    assert torch.isfinite(sae_output).all()
    assert (sae_output >= 0).all()  # JumpReLU

    rows = []
    for position, (token_id, values) in enumerate(zip(token_ids, sae_output[0].cpu(), strict=True)):
        top_values, top_ids = values.topk(k=top_k)
        row = {
            "position": position,
            "token_id": token_id,
            "token": tokenizer.convert_ids_to_tokens(token_id),
            "n_active": (values > 0).sum().item(),
        }
        for rank, (feature_id, activation) in enumerate(zip(top_ids.tolist(), top_values.tolist(), strict=True), start=1):
            row[f"top_{rank}_feature"] = (feature_id, activation) if activation > 0 else None
        rows.append(row)

    df = pd.DataFrame(rows)
    feature_columns = [f"top_{rank}_feature" for rank in range(1, top_k + 1)]
    assert df["position"].tolist() == list(range(len(token_ids)))
    assert df["token_id"].tolist() == token_ids
    assert df["n_active"].sum() == torch.count_nonzero(sae_output).item()
    assert df[feature_columns].notna().sum(axis=1).eq(df["n_active"].clip(upper=top_k)).all()
    return df


In [45]:
get_sae_features(messages)

,position,token_id,token,n_active,top_1_feature,top_2_feature,top_3_feature,top_4_feature,top_5_feature
0,0,2,<bos>,132,"(419, 4932.583984375)","(115, 4566.9921875)","(206, 3911.41357421875)","(502, 3310.19140625)","(257, 3233.30029296875)"
1,1,105,<start_of_turn>,65,"(371, 1134.3150634765625)","(698, 559.78955078125)","(96, 395.5174865722656)","(964, 328.7417297363281)","(131, 310.14031982421875)"
2,2,2364,user,36,"(363, 474.01727294921875)","(206, 437.98797607421875)","(544, 419.199951171875)","(1010, 369.8804626464844)","(613, 345.8411560058594)"
3,3,107,\n,53,"(419, 1661.5394287109375)","(206, 1482.8916015625)","(115, 1461.299560546875)","(257, 1076.557373046875)","(502, 1073.655029296875)"
4,4,155122,Explain,33,"(374, 473.3941650390625)","(1259, 417.5339050292969)","(62, 413.04052734375)","(446, 320.7629699707031)","(313, 267.2330017089844)"
5,5,528,▁in,33,"(374, 431.1261291503906)","(613, 271.51702880859375)","(898, 251.61434936523438)","(245, 187.1352081298828)","(62, 179.950439453125)"
6,6,1156,▁two,51,"(374, 410.51580810546875)","(425, 279.9611511230469)","(2131, 267.32513427734375)","(241, 197.0941619873047)","(2732, 194.9609832763672)"
7,7,23974,▁sentences,60,"(793, 413.11956787109375)","(374, 328.4774475097656)","(47, 316.62261962890625)","(222, 292.6207275390625)","(6954, 284.687255859375)"
8,8,3217,▁why,31,"(374, 431.7164001464844)","(736, 402.97869873046875)","(62, 328.367431640625)","(303, 286.6940002441406)","(14188, 235.63113403320312)"
9,9,506,▁the,27,"(396, 575.0133056640625)","(374, 406.9334716796875)","(1211, 226.64437866210938)","(62, 185.25497436523438)","(736, 154.5821533203125)"


In [52]:
get_sae_features([{"role": "user", "content": "cat"}]).iloc[4]

position                                  4
token_id                               9307
token                                   cat
n_active                                 77
top_1_feature      (291, 343.5960693359375)
top_2_feature     (374, 291.42547607421875)
top_3_feature    (1211, 271.50323486328125)
top_4_feature     (387, 229.11810302734375)
top_5_feature     (131, 223.65292358398438)
Name: 4, dtype: object

In [51]:
get_sae_features([{"role": "user", "content": "kitten"}]).iloc[4]

position                                  4
token_id                             231775
token                                kitten
n_active                                 90
top_1_feature     (241, 342.87884521484375)
top_2_feature    (1211, 226.71099853515625)
top_3_feature      (374, 214.6046600341797)
top_4_feature     (291, 211.94509887695312)
top_5_feature     (8908, 176.7518768310547)
Name: 4, dtype: object

In [53]:
get_sae_features([{"role": "user", "content": "dog"}]).iloc[4]

position                                  4
token_id                              21029
token                                   dog
n_active                                 78
top_1_feature      (374, 300.7938537597656)
top_2_feature    (8908, 294.83074951171875)
top_3_feature     (1211, 276.8858947753906)
top_4_feature       (241, 270.972900390625)
top_5_feature     (291, 181.14273071289062)
Name: 4, dtype: object

Гипотеза: $374$ – фича "животное".

Проверяем **селективность** 374: восемь названий животных против предметов, абстрактных слов и глаголов; каждое слово отдельно и в шаблоне `Describe the {word}.`. Измеряем саму координату 374, даже если она не попала в top-5. Усредняем по токенам целевого слова; служебные токены и остальные слова шаблона исключаем. `active_fraction` — доля примеров хотя бы с одной положительной активацией на целевом слове. Глаголы во втором шаблоне — дополнительный, грамматически неоднородный контроль; основное сравнение — животные и предметы.

In [54]:
FEATURE_ID = 374

@torch.inference_mode()
def measure_feature_374(text, target):
    """Активация 374 на всех токенах целевого слова, без chat template."""
    rendered = tokenizer.apply_chat_template(
        [{"role": "user", "content": text}],
        tokenize=False, add_generation_prompt=True,
    )
    assert text.count(target) == 1 and rendered.count(text) == 1
    start = rendered.index(text) + text.index(target)
    end = start + len(target)
    encoded = tokenizer(
        rendered, add_special_tokens=False, return_offsets_mapping=True,
        return_tensors="pt",
    )
    offsets = encoded.pop("offset_mapping")[0].tolist()
    encoded = encoded.to(model.device)
    positions = [i for i, (a, b) in enumerate(offsets) if b > start and a < end]
    assert positions

    # Хук только на resid_post; снимаем даже при ошибке.
    captured = {}
    def save_post(module, args, output):
        hidden = output[0] if isinstance(output, tuple) else output
        captured["post"] = hidden[:, positions, :].detach().clone()

    handle = block.register_forward_hook(save_post)
    try:
        model.model(**encoded, use_cache=False)
    finally:
        handle.remove()

    hidden = captured["post"]
    assert hidden.shape == (1, len(positions), sae.cfg.d_in)
    assert torch.isfinite(hidden).all()
    features = sae.encode(hidden.to(device=sae.device, dtype=sae.dtype))
    assert features.shape == (1, len(positions), sae.cfg.d_sae)
    assert torch.isfinite(features).all() and (features >= 0).all()
    values = features[0, :, FEATURE_ID].cpu()
    ids = encoded["input_ids"][0, positions].tolist()
    return {
        "tokens": tokenizer.convert_ids_to_tokens(ids),
        "activations": values.tolist(),
        "mean_374": values.mean().item(),
        "max_374": values.max().item(),
        "active": bool((values > 0).any()),
    }


# Заранее заданные контроли; cat/kitten/dog — исходные примеры.
probe_words = {
    "animal": ["cat", "kitten", "dog", "horse", "fish", "eagle", "snake", "butterfly"],
    "object": ["chair", "table", "pencil", "bottle", "computer", "stone", "bicycle", "camera"],
    "abstract": ["love", "justice", "music", "science", "freedom", "number", "truth", "memory"],
    "verb": ["explain", "write", "run", "think", "build", "sleep", "read", "calculate"],
}
probe_templates = {"isolated": "{word}", "sentence": "Describe the {word}."}
probe_rows = []
for context, template in probe_templates.items():
    for group, words in probe_words.items():
        for word in words:
            text = template.format(word=word)
            probe_rows.append({
                "context": context, "group": group, "word": word,
                **measure_feature_374(text, word),
            })
    print(f"{context}: {sum(row['context'] == context for row in probe_rows)} prompts done")

feature_374_probe = pd.DataFrame(probe_rows)
assert len(feature_374_probe) == 64
assert not feature_374_probe.duplicated(["context", "word"]).any()
display(feature_374_probe.groupby(["context", "group"], sort=False).agg(
    n=("word", "size"),
    mean_374=("mean_374", "mean"),
    median_374=("mean_374", "median"),
    active_fraction=("active", "mean"),
).round(2))
display(feature_374_probe.pivot(index=["group", "word"], columns="context", values="mean_374").round(2))


isolated: 32 prompts done
sentence: 32 prompts done


n  mean_374  median_374  active_fraction
context  group                                             
isolated animal    8    272.21      289.28              1.0
         object    8    298.17      299.68              1.0
         abstract  8    353.32      374.58              1.0
         verb      8    462.38      485.90              1.0
sentence animal    8    133.54      124.90              1.0
         object    8    192.44      218.78              1.0
         abstract  8    293.91      304.76              1.0
         verb      8    373.87      409.96              1.0

context             isolated  sentence
group    word                         
abstract freedom      297.85    275.60
         justice      249.00    203.68
         love         408.68    271.09
         memory       383.60    333.92
         music        305.42    191.38
         number       429.00    383.21
         science      387.44    352.57
         truth        365.55    339.83
animal   butterfly    191.72    102.90
         cat          291.43    117.26
         dog          300.79    182.95
         eagle        352.21     82.17
         fish         305.93    187.88
         horse        233.89    132.54
         kitten       214.60    101.32
         snake        287.13    161.32
object   bicycle      265.05     95.69
         bottle       288.01    220.71
         camera       325.47    216.84
         chair        230.08     78.74
         computer     372.53    281.82
         pencil       238.65     65.29
         stone        354.18    270.79
         table        311.35    309.62
verb     build        498.38    411.58
         calculate    492.56    431.39
         explain      473.08    442.80
         read         487.01    400.70
         run          470.27    368.69
         sleep        288.09    110.52
         think        504.88    408.34
         write        484.79    416.93

**Гипотеза не поддерживается этой проверкой.** 374 активна во всех 64 примерах, включая все контроли. Средние активации для животных / предметов / абстрактных слов / глаголов: **272 / 298 / 353 / 462** для отдельных слов и **134 / 192 / 294 / 374** в предложении. Животные не дают повышенного отклика ни в одном контексте; наличие 374 в top-5 у `cat`, `kitten` и `dog` само по себе не означает «животное».

Это небольшой вручную заданный набор: он показывает неселективность на проверенных примерах, но не устанавливает точный смысл фичи и ничего не говорит о её каузальном эффекте.

# Итого

Мы приходим к тому, что интерпретация, и тем более стиринг – сложная задача. Так с наскоку не взять :)